# Machine Translation using Hugging Face (English to Hindi)


## Part A: Setting Up Machine Translation Model



In [1]:
import warnings
warnings.filterwarnings("ignore")

import torch
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-en-hi"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

### Translation Function


In [2]:
def translate_sentence(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        translated_tokens = model.generate(**inputs, max_length=128)
    translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
    print(f"Source Sentence: {text}")
    print(f"Translated Sentence: {translated_text}\n")
    return translated_text

### Model Testing with Diverse Sentences

In [3]:
sample_sentences = [
    "Machine learning algorithms learn patterns directly from data.",
    "The library is open every day except Sunday.",
    "Water is essential for all known forms of life.",
    "She plays the piano with great passion.",
    "Education empowers individuals to achieve their goals."
]

for sentence in sample_sentences:
    translate_sentence(sentence)

Source Sentence: Machine learning algorithms learn patterns directly from data.
Translated Sentence: मशीन सीखने का एल्गोरिदम सीधे डेटा से पैटर्न सीखते हैं.



Source Sentence: The library is open every day except Sunday.
Translated Sentence: पुस्तकालय रविवार को छोड़कर हर दिन खुला हुआ है.



Source Sentence: Water is essential for all known forms of life.
Translated Sentence: जीवन के सभी ज्ञात रूपों के लिए जल अनिवार्य है ।



Source Sentence: She plays the piano with great passion.
Translated Sentence: वह बड़ी वासना के साथ पियानो खेलता है.



Source Sentence: Education empowers individuals to achieve their goals.
Translated Sentence: शिक्षा लोगों को अपने लक्ष्य हासिल करने में मदद देती है ।



## Part B: Custom Dataset Translation

In [4]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

dataset_sentences = [
    "Artificial intelligence is transforming modern healthcare.",
    "She reads books in the library every afternoon.",
    "The sun rises in the east and sets in the west.",
    "We should protect the environment for future generations.",
    "He decided to study computer science at the university.",
    "The train arrived at the station on time today.",
    "Good health is the greatest wealth in life.",
    "The chef prepared a delicious dinner for the guests.",
    "Honesty is considered the best policy across cultures.",
    "Students are preparing diligently for their upcoming examinations."
]

translations = []
for text in dataset_sentences:
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        output = model.generate(**inputs, max_length=128)
    translated = tokenizer.decode(output[0], skip_special_tokens=True)
    translations.append(translated)

df = pd.DataFrame({
    "Source Sentence": dataset_sentences,
    "Translated Sentence": translations
})

df.to_csv("translations.csv", index=False, encoding="utf-8-sig")
df

,Source Sentence,Translated Sentence
0,Artificial intelligence is transforming modern healthcare.,विज्ञान - विज्ञान आज की सेहत के बारे में भी कुछ ऐसा ही सोच रहा है ।
1,She reads books in the library every afternoon.,वह हर दोपहर पुस्तकालय में किताबें पढ़ती है ।
2,The sun rises in the east and sets in the west.,पूर्व में सूरज उदय होता है और पश्चिम में सेट.
3,We should protect the environment for future generations.,हमें आनेवाली पीढ़ियों के वातावरण की रक्षा करनी चाहिए ।
4,He decided to study computer science at the university.,उसने विश्‍वविद्यालय में कंप्यूटर विज्ञान का अध्ययन करने का फैसला किया ।
5,The train arrived at the station on time today.,आज ट्रेन उस स्टेशन पर पहुँच गयी है ।
6,Good health is the greatest wealth in life.,अच्छा स्वास्थ्य जीवन में सबसे बड़ा धन है ।
7,The chef prepared a delicious dinner for the guests.,उन्होंने मेहमानों के लिए एक स्वादिष्ट दावत तैयार की ।
8,Honesty is considered the best policy across cultures.,ईमानदारी को संस्कृतियों के दौरान सबसे बेहतरीन नीति माना जाता है ।
9,Students are preparing diligently for their upcoming examinations.,विद्यार्थी अपनी भविष्य की जाँच करने के लिए जी - जान से मेहनत कर रहे हैं ।


## Part C: Model Performance Evaluation

In [5]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction

reference_translations = [
    "कृत्रिम बुद्धिमत्ता आधुनिक स्वास्थ्य सेवा को बदल रही है।",
    "वह हर दोपहर पुस्तकालय में किताबें पढ़ती है।",
    "सूरज पूरब में उगता है और पश्चिम में अस्त होता है।",
    "हमें भविष्य की पीढ़ियों के लिए पर्यावरण की रक्षा करनी चाहिए।",
    "उसने विश्वविद्यालय में कंप्यूटर विज्ञान पढ़ने का निर्णय लिया।",
    "आज ट्रेन समय पर स्टेशन पहुंची।",
    "अच्छा स्वास्थ्य जीवन का सबसे बड़ा धन है।",
    "रसोइये ने मेहमानों के लिए स्वादिष्ट रात्रिभोज तैयार किया।",
    "ईमानदारी को सभी संस्कृतियों में सबसे अच्छी नीति माना जाता है।",
    "छात्र अपनी आगामी परीक्षाओं के लिए लगन से तैयारी कर रहे हैं।"
]

ref_tokens = [[ref.split()] for ref in reference_translations]
pred_tokens = [pred.split() for pred in translations]

smoothie = SmoothingFunction().method1

sentence_bleu_scores = []
for ref, pred in zip(ref_tokens, pred_tokens):
    score = sentence_bleu(ref, pred, smoothing_function=smoothie)
    sentence_bleu_scores.append(round(score, 4))

overall_corpus_bleu = corpus_bleu(ref_tokens, pred_tokens, smoothing_function=smoothie)

eval_df = pd.DataFrame({
    "Source Sentence": dataset_sentences,
    "Reference Translation": reference_translations,
    "Model Translation": translations,
    "BLEU Score": sentence_bleu_scores
})

print(f"Overall Corpus BLEU Score: {overall_corpus_bleu:.4f} ({overall_corpus_bleu * 100:.2f}%)\n")
eval_df

Overall Corpus BLEU Score: 0.1741 (17.41%)



,Source Sentence,Reference Translation,Model Translation,BLEU Score
0,Artificial intelligence is transforming modern healthcare.,कृत्रिम बुद्धिमत्ता आधुनिक स्वास्थ्य सेवा को बदल रही है।,विज्ञान - विज्ञान आज की सेहत के बारे में भी कुछ ऐसा ही सोच रहा है ।,0.0000
1,She reads books in the library every afternoon.,वह हर दोपहर पुस्तकालय में किताबें पढ़ती है।,वह हर दोपहर पुस्तकालय में किताबें पढ़ती है ।,0.7260
2,The sun rises in the east and sets in the west.,सूरज पूरब में उगता है और पश्चिम में अस्त होता है।,पूर्व में सूरज उदय होता है और पश्चिम में सेट.,0.2734
3,We should protect the environment for future generations.,हमें भविष्य की पीढ़ियों के लिए पर्यावरण की रक्षा करनी चाहिए।,हमें आनेवाली पीढ़ियों के वातावरण की रक्षा करनी चाहिए ।,0.1244
4,He decided to study computer science at the university.,उसने विश्वविद्यालय में कंप्यूटर विज्ञान पढ़ने का निर्णय लिया।,उसने विश्‍वविद्यालय में कंप्यूटर विज्ञान का अध्ययन करने का फैसला किया ।,0.0958
5,The train arrived at the station on time today.,आज ट्रेन समय पर स्टेशन पहुंची।,आज ट्रेन उस स्टेशन पर पहुँच गयी है ।,0.0603
6,Good health is the greatest wealth in life.,अच्छा स्वास्थ्य जीवन का सबसे बड़ा धन है।,अच्छा स्वास्थ्य जीवन में सबसे बड़ा धन है ।,0.1996
7,The chef prepared a delicious dinner for the guests.,रसोइये ने मेहमानों के लिए स्वादिष्ट रात्रिभोज तैयार किया।,उन्होंने मेहमानों के लिए एक स्वादिष्ट दावत तैयार की ।,0.1187
8,Honesty is considered the best policy across cultures.,ईमानदारी को सभी संस्कृतियों में सबसे अच्छी नीति माना जाता है।,ईमानदारी को संस्कृतियों के दौरान सबसे बेहतरीन नीति माना जाता है ।,0.1153
9,Students are preparing diligently for their upcoming examinations.,छात्र अपनी आगामी परीक्षाओं के लिए लगन से तैयारी कर रहे हैं।,विद्यार्थी अपनी भविष्य की जाँच करने के लिए जी - जान से मेहनत कर रहे हैं ।,0.0381


### Discussion of BLEU Score Results

The model achieved an overall corpus BLEU score of **0.1741 (17.41%)** across the 10 evaluation sentences.

Due to the nature of BLUE score, and hindi being a highly morphological language, BLUE score is coming less but still the translations are actually good enough.

### Discrepancy & Error Analysis

---

#### Example 1: Domain-Specific Terminology and Meaning Distortion (Sentence 1)
- **Source Sentence:** *"Artificial intelligence is transforming modern healthcare."*
- **Model Translation:** `"विज्ञान - विज्ञान आज की सेहत के बारे में भी कुछ ऐसा ही सोच रहा है ।"`
- **Reference Translation:** `"कृत्रिम बुद्धिमत्ता आधुनिक स्वास्थ्य सेवा को बदल रही है।"`


#### Example 2: Partial Transliteration & Untranslated Loanword (Sentence 3)
- **Source Sentence:** *"The sun rises in the east and sets in the west."*
- **Model Translation:** `"पूर्व में सूरज उदय होता है और पश्चिम में सेट."`
- **Reference Translation:** `"सूरज पूरब में उगता है और पश्चिम में अस्त होता है।"`